In [1]:
# Cell 1. Load selected pre-sleep final model predictions for uncertainty analysis
# 원하는 결과:
# - 최종 대표 모델 prediction을 로드한다.
# - selected experiment/test split만 추출한다.
# - participant-level bootstrap 준비 상태를 확인한다.
# - 아직 bootstrap은 실행하지 않는다.

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    brier_score_loss,
)

PROJECT_ROOT = Path(r"C:\workSpace\DeepLearnin_sleep")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_PATH = PROJECT_ROOT / "log" / "2026-06-29.md"

STAGE1_DIR = PROCESSED_DIR / "pre_sleep_forecasting" / "design_c_stage1"
STABILITY_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_hyperparameter_stability_outputs"

METADATA_PATH = STAGE1_DIR / "metadata.json"
PREDICTIONS_PATH = STABILITY_OUTPUT_DIR / "stage1_hyperparameter_stability_predictions.csv"
SELECTION_SUMMARY_PATH = STABILITY_OUTPUT_DIR / "stage1_best_config_seed_selection_summary.csv"
THRESHOLD_POLICY_PATH = STABILITY_OUTPUT_DIR / "stage1_selected_model_threshold_policy_comparison.csv"

UNCERTAINTY_OUTPUT_DIR = STAGE1_DIR / "experiments" / "stage1_final_uncertainty_calibration_outputs"
UNCERTAINTY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
selection_summary_df = pd.read_csv(SELECTION_SUMMARY_PATH, encoding="utf-8-sig")
threshold_policy_df = pd.read_csv(THRESHOLD_POLICY_PATH, encoding="utf-8-sig")
predictions_df = pd.read_csv(PREDICTIONS_PATH, encoding="utf-8-sig")

selected_row = selection_summary_df[
    selection_summary_df["selection_role"] == "selected_by_validation"
].iloc[0]

SELECTED_EXPERIMENT_ID = selected_row["experiment_id"]
SELECTED_THRESHOLD = float(selected_row["selected_threshold_from_validation"])

selected_test_predictions_df = predictions_df[
    (predictions_df["experiment_id"] == SELECTED_EXPERIMENT_ID)
    & (predictions_df["split"] == "test")
].copy()

selected_test_predictions_df["y_pred_selected_threshold"] = (
    selected_test_predictions_df["y_probability"] >= SELECTED_THRESHOLD
).astype(int)

def evaluate_prediction_frame(prediction_df, threshold):
    y_true = prediction_df["y_true"].to_numpy()
    y_probability = prediction_df["y_probability"].to_numpy()
    y_pred = (y_probability >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "samples": int(len(prediction_df)),
        "participants": int(prediction_df["participant_object_id"].nunique()),
        "threshold": float(threshold),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_probability),
        "average_precision": average_precision_score(y_true, y_probability),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, y_probability),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

point_metrics = evaluate_prediction_frame(
    selected_test_predictions_df,
    SELECTED_THRESHOLD,
)

participant_prediction_summary = (
    selected_test_predictions_df
    .groupby("participant_object_id")
    .agg(
        rows=("y_true", "size"),
        target_mean=("y_true", "mean"),
        probability_mean=("y_probability", "mean"),
        predicted_positive_rate=("y_pred_selected_threshold", "mean"),
    )
    .reset_index()
)

print("=== Selected Final Model Prediction Setup ===")
print("selected experiment:", SELECTED_EXPERIMENT_ID)
print("selected threshold:", SELECTED_THRESHOLD)
print("prediction rows:", len(selected_test_predictions_df))
print("test participants:", selected_test_predictions_df["participant_object_id"].nunique())

print("\n=== Point Test Metrics ===")
display(pd.DataFrame([point_metrics]))

print("\n=== Participant Prediction Summary ===")
display(participant_prediction_summary)

print("\n=== Bootstrap Readiness Checks ===")
print("missing y_true:", int(selected_test_predictions_df["y_true"].isna().sum()))
print("missing y_probability:", int(selected_test_predictions_df["y_probability"].isna().sum()))
print("unique y_true values:", sorted(selected_test_predictions_df["y_true"].unique().tolist()))
print("participants with one class only:", int(((participant_prediction_summary["target_mean"] == 0) | (participant_prediction_summary["target_mean"] == 1)).sum()))

print("\nReady for next step: participant-level bootstrap confidence intervals.")

=== Selected Final Model Prediction Setup ===
selected experiment: presleep_stage1_hp_tiny_dropout40_wd1e3_seed2027
selected threshold: 0.54
prediction rows: 881
test participants: 14

=== Point Test Metrics ===


,samples,participants,threshold,balanced_accuracy,roc_auc,average_precision,f1,precision,recall,brier_score,tn,fp,fn,tp
0,881,14,0.54,0.649209,0.693695,0.618684,0.515267,0.65534,0.424528,0.212638,492,71,183,135



=== Participant Prediction Summary ===


,participant_object_id,rows,target_mean,probability_mean,predicted_positive_rate
0,621e2f3967b776a240c654db,62,0.612903,0.538163,0.500000
1,621e2f6167b776a240e082a9,45,0.288889,0.365306,0.044444
2,621e2fb367b776a24015accd,59,0.254237,0.360955,0.016949
3,621e2ff067b776a2403eb737,60,0.716667,0.519048,0.483333
4,621e301e67b776a240608a72,53,0.452830,0.364146,0.094340
5,621e30e467b776a240e817c7,30,0.500000,0.447506,0.166667
6,621e30f467b776a240f22944,62,0.516129,0.460110,0.322581
7,621e310d67b776a24003096d,67,0.388060,0.454638,0.149254
8,621e324e67b776a2400191cb,68,0.102941,0.376562,0.029412
9,621e32e667b776a2406d2f1c,90,0.222222,0.526210,0.411111



=== Bootstrap Readiness Checks ===
missing y_true: 0
missing y_probability: 0
unique y_true values: [0, 1]
participants with one class only: 0

Ready for next step: participant-level bootstrap confidence intervals.


In [2]:
# Cell 2. Participant-level bootstrap confidence intervals for selected final model
# 원하는 결과:
# - test participant를 단위로 bootstrap resampling한다.
# - selected threshold=0.54 기준 주요 metric의 95% CI를 계산한다.
# - ROC AUC/AP/Brier score도 함께 계산한다.
# - bootstrap 결과와 summary를 저장하고 log를 업데이트한다.

BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_RANDOM_STATE = 2026

BOOTSTRAP_DISTRIBUTION_PATH = UNCERTAINTY_OUTPUT_DIR / "selected_model_participant_bootstrap_distribution.csv"
BOOTSTRAP_SUMMARY_PATH = UNCERTAINTY_OUTPUT_DIR / "selected_model_participant_bootstrap_summary.csv"

rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)

test_participants = selected_test_predictions_df["participant_object_id"].drop_duplicates().to_numpy()

bootstrap_rows = []

print("=== Participant-Level Bootstrap Started ===")
print("iterations:", BOOTSTRAP_ITERATIONS)
print("participants:", len(test_participants))
print("threshold:", SELECTED_THRESHOLD)

for iteration in range(BOOTSTRAP_ITERATIONS):
    sampled_participants = rng.choice(
        test_participants,
        size=len(test_participants),
        replace=True,
    )

    sampled_frames = []

    for draw_index, participant_id in enumerate(sampled_participants):
        participant_rows = selected_test_predictions_df[
            selected_test_predictions_df["participant_object_id"] == participant_id
        ].copy()
        participant_rows["bootstrap_draw_index"] = draw_index
        sampled_frames.append(participant_rows)

    bootstrap_df = pd.concat(sampled_frames, ignore_index=True)

    y_true = bootstrap_df["y_true"].to_numpy()
    y_probability = bootstrap_df["y_probability"].to_numpy()
    y_pred = (y_probability >= SELECTED_THRESHOLD).astype(int)

    row = {
        "iteration": iteration,
        "samples": int(len(bootstrap_df)),
        "unique_participants": int(pd.Series(sampled_participants).nunique()),
        "target_mean": float(y_true.mean()),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "brier_score": brier_score_loss(y_true, y_probability),
    }

    if len(np.unique(y_true)) == 2:
        row["roc_auc"] = roc_auc_score(y_true, y_probability)
        row["average_precision"] = average_precision_score(y_true, y_probability)
    else:
        row["roc_auc"] = np.nan
        row["average_precision"] = np.nan

    bootstrap_rows.append(row)

    if (iteration + 1) % 1000 == 0:
        print(f"{iteration + 1}/{BOOTSTRAP_ITERATIONS} bootstrap iterations complete")

bootstrap_distribution_df = pd.DataFrame(bootstrap_rows)

metric_columns = [
    "balanced_accuracy",
    "roc_auc",
    "average_precision",
    "f1",
    "precision",
    "recall",
    "brier_score",
]

summary_rows = []

for metric in metric_columns:
    values = bootstrap_distribution_df[metric].dropna()

    summary_rows.append(
        {
            "metric": metric,
            "point_estimate": point_metrics[metric],
            "bootstrap_mean": values.mean(),
            "bootstrap_std": values.std(ddof=1),
            "ci_lower_2_5": values.quantile(0.025),
            "ci_upper_97_5": values.quantile(0.975),
            "bootstrap_non_missing_iterations": int(values.shape[0]),
        }
    )

bootstrap_summary_df = pd.DataFrame(summary_rows)

bootstrap_distribution_df.to_csv(
    BOOTSTRAP_DISTRIBUTION_PATH,
    index=False,
    encoding="utf-8-sig",
)
bootstrap_summary_df.to_csv(
    BOOTSTRAP_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

metadata["final_model_participant_bootstrap"] = {
    "selected_experiment_id": SELECTED_EXPERIMENT_ID,
    "selected_threshold": SELECTED_THRESHOLD,
    "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
    "bootstrap_random_state": BOOTSTRAP_RANDOM_STATE,
    "participant_count": int(len(test_participants)),
    "distribution_path": str(BOOTSTRAP_DISTRIBUTION_PATH.relative_to(PROJECT_ROOT)),
    "summary_path": str(BOOTSTRAP_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
}

METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

ba_summary = bootstrap_summary_df[
    bootstrap_summary_df["metric"] == "balanced_accuracy"
].iloc[0]

log_entry = f"""

## 2026-06-29 - Selected pre-sleep model participant bootstrap CI

Computed participant-level bootstrap uncertainty for selected final pre-sleep model.

- Selected experiment: `{SELECTED_EXPERIMENT_ID}`
- Threshold: {SELECTED_THRESHOLD:.2f}
- Test participants: {len(test_participants)}
- Bootstrap iterations: {BOOTSTRAP_ITERATIONS}
- Bootstrap distribution: `{BOOTSTRAP_DISTRIBUTION_PATH.relative_to(PROJECT_ROOT)}`
- Bootstrap summary: `{BOOTSTRAP_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- Balanced accuracy point estimate: {ba_summary["point_estimate"]:.4f}
- Balanced accuracy bootstrap 95% CI: [{ba_summary["ci_lower_2_5"]:.4f}, {ba_summary["ci_upper_97_5"]:.4f}]
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("\n=== Participant-Level Bootstrap Saved ===")
print("distribution:", BOOTSTRAP_DISTRIBUTION_PATH.relative_to(PROJECT_ROOT), BOOTSTRAP_DISTRIBUTION_PATH.exists())
print("summary:", BOOTSTRAP_SUMMARY_PATH.relative_to(PROJECT_ROOT), BOOTSTRAP_SUMMARY_PATH.exists())

print("\n=== Bootstrap Summary ===")
display(bootstrap_summary_df)

print("\n=== Bootstrap Distribution Preview ===")
display(bootstrap_distribution_df.head())

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Participant-Level Bootstrap Started ===
iterations: 5000
participants: 14
threshold: 0.54
1000/5000 bootstrap iterations complete
2000/5000 bootstrap iterations complete
3000/5000 bootstrap iterations complete
4000/5000 bootstrap iterations complete
5000/5000 bootstrap iterations complete

=== Participant-Level Bootstrap Saved ===
distribution: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_final_uncertainty_calibration_outputs\selected_model_participant_bootstrap_distribution.csv True
summary: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_final_uncertainty_calibration_outputs\selected_model_participant_bootstrap_summary.csv True

=== Bootstrap Summary ===


,metric,point_estimate,bootstrap_mean,bootstrap_std,ci_lower_2_5,ci_upper_97_5,bootstrap_non_missing_iterations
0,balanced_accuracy,0.649209,0.642924,0.047049,0.543646,0.725915,5000
1,roc_auc,0.693695,0.684564,0.059439,0.556669,0.787220,5000
2,average_precision,0.618684,0.607106,0.103430,0.373546,0.777308,5000
3,f1,0.515267,0.499318,0.100004,0.271381,0.660956,5000
4,precision,0.655340,0.644111,0.120576,0.373966,0.837727,5000
5,recall,0.424528,0.411942,0.092917,0.205422,0.567629,5000
6,brier_score,0.212638,0.212790,0.010718,0.192430,0.234474,5000



=== Bootstrap Distribution Preview ===


,iteration,samples,unique_participants,target_mean,balanced_accuracy,f1,precision,recall,brier_score,roc_auc,average_precision
0,0,804,10,0.277363,0.578140,0.370927,0.420455,0.331839,0.225493,0.586518,0.427341
1,1,1018,9,0.358546,0.698498,0.591736,0.745833,0.490411,0.201970,0.736722,0.655940
2,2,848,8,0.369104,0.585847,0.430189,0.525346,0.364217,0.231348,0.603720,0.549345
3,3,820,10,0.367073,0.673359,0.555332,0.704082,0.458472,0.206953,0.711591,0.661993
4,4,891,9,0.249158,0.529501,0.216718,0.346535,0.157658,0.209589,0.547873,0.340315



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [3]:
# Cell 3. Calibration analysis for selected final pre-sleep model
# 원하는 결과:
# - selected final model의 test probability calibration을 평가한다.
# - probability bin별 predicted probability와 observed event rate를 계산한다.
# - ECE/MCE/Brier score를 계산한다.
# - calibration table/summary를 저장하고 log를 업데이트한다.

CALIBRATION_BINS = 10

CALIBRATION_TABLE_PATH = UNCERTAINTY_OUTPUT_DIR / "selected_model_calibration_table.csv"
CALIBRATION_SUMMARY_PATH = UNCERTAINTY_OUTPUT_DIR / "selected_model_calibration_summary.csv"

calibration_df = selected_test_predictions_df.copy()
calibration_df["probability_bin"] = pd.cut(
    calibration_df["y_probability"],
    bins=np.linspace(0, 1, CALIBRATION_BINS + 1),
    include_lowest=True,
)

calibration_table_df = (
    calibration_df
    .groupby("probability_bin", observed=False)
    .agg(
        samples=("y_true", "size"),
        mean_predicted_probability=("y_probability", "mean"),
        observed_positive_rate=("y_true", "mean"),
        predicted_positive_rate_at_threshold=("y_pred_selected_threshold", "mean"),
    )
    .reset_index()
)

calibration_table_df["probability_bin"] = calibration_table_df[
    "probability_bin"
].astype(str)

calibration_table_df["abs_calibration_error"] = (
    calibration_table_df["mean_predicted_probability"]
    - calibration_table_df["observed_positive_rate"]
).abs()

non_empty_bins = calibration_table_df[calibration_table_df["samples"] > 0].copy()
total_samples = non_empty_bins["samples"].sum()

ece = (
    non_empty_bins["samples"]
    / total_samples
    * non_empty_bins["abs_calibration_error"]
).sum()

mce = non_empty_bins["abs_calibration_error"].max()

brier = brier_score_loss(
    selected_test_predictions_df["y_true"],
    selected_test_predictions_df["y_probability"],
)

calibration_summary_df = pd.DataFrame(
    [
        {
            "metric": "samples",
            "value": len(selected_test_predictions_df),
        },
        {
            "metric": "participants",
            "value": selected_test_predictions_df["participant_object_id"].nunique(),
        },
        {
            "metric": "bins",
            "value": CALIBRATION_BINS,
        },
        {
            "metric": "brier_score",
            "value": brier,
        },
        {
            "metric": "expected_calibration_error",
            "value": ece,
        },
        {
            "metric": "maximum_calibration_error",
            "value": mce,
        },
        {
            "metric": "mean_predicted_probability",
            "value": selected_test_predictions_df["y_probability"].mean(),
        },
        {
            "metric": "observed_positive_rate",
            "value": selected_test_predictions_df["y_true"].mean(),
        },
        {
            "metric": "selected_threshold",
            "value": SELECTED_THRESHOLD,
        },
    ]
)

participant_calibration_df = (
    selected_test_predictions_df
    .groupby("participant_object_id")
    .agg(
        samples=("y_true", "size"),
        observed_positive_rate=("y_true", "mean"),
        mean_predicted_probability=("y_probability", "mean"),
        brier_score=("y_probability", lambda p: brier_score_loss(
            selected_test_predictions_df.loc[p.index, "y_true"],
            p,
        )),
    )
    .reset_index()
)

participant_calibration_df["abs_mean_probability_error"] = (
    participant_calibration_df["mean_predicted_probability"]
    - participant_calibration_df["observed_positive_rate"]
).abs()

calibration_table_df.to_csv(
    CALIBRATION_TABLE_PATH,
    index=False,
    encoding="utf-8-sig",
)
calibration_summary_df.to_csv(
    CALIBRATION_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

PARTICIPANT_CALIBRATION_PATH = UNCERTAINTY_OUTPUT_DIR / "selected_model_participant_calibration.csv"
participant_calibration_df.to_csv(
    PARTICIPANT_CALIBRATION_PATH,
    index=False,
    encoding="utf-8-sig",
)

metadata["final_model_calibration"] = {
    "selected_experiment_id": SELECTED_EXPERIMENT_ID,
    "selected_threshold": SELECTED_THRESHOLD,
    "bins": CALIBRATION_BINS,
    "brier_score": float(brier),
    "expected_calibration_error": float(ece),
    "maximum_calibration_error": float(mce),
    "calibration_table_path": str(CALIBRATION_TABLE_PATH.relative_to(PROJECT_ROOT)),
    "calibration_summary_path": str(CALIBRATION_SUMMARY_PATH.relative_to(PROJECT_ROOT)),
    "participant_calibration_path": str(PARTICIPANT_CALIBRATION_PATH.relative_to(PROJECT_ROOT)),
}

METADATA_PATH.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

log_entry = f"""

## 2026-06-29 - Selected pre-sleep model calibration analysis

Computed calibration diagnostics for selected final pre-sleep model.

- Selected experiment: `{SELECTED_EXPERIMENT_ID}`
- Threshold: {SELECTED_THRESHOLD:.2f}
- Brier score: {brier:.4f}
- Expected calibration error: {ece:.4f}
- Maximum calibration error: {mce:.4f}
- Calibration table: `{CALIBRATION_TABLE_PATH.relative_to(PROJECT_ROOT)}`
- Calibration summary: `{CALIBRATION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}`
- Participant calibration: `{PARTICIPANT_CALIBRATION_PATH.relative_to(PROJECT_ROOT)}`
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("=== Calibration Analysis Saved ===")
print("calibration table:", CALIBRATION_TABLE_PATH.relative_to(PROJECT_ROOT), CALIBRATION_TABLE_PATH.exists())
print("calibration summary:", CALIBRATION_SUMMARY_PATH.relative_to(PROJECT_ROOT), CALIBRATION_SUMMARY_PATH.exists())
print("participant calibration:", PARTICIPANT_CALIBRATION_PATH.relative_to(PROJECT_ROOT), PARTICIPANT_CALIBRATION_PATH.exists())

print("\n=== Calibration Summary ===")
display(calibration_summary_df)

print("\n=== Calibration Table ===")
display(calibration_table_df)

print("\n=== Participant Calibration Summary ===")
display(
    participant_calibration_df
    .sort_values("abs_mean_probability_error", ascending=False)
)

print("\nlog updated:", LOG_PATH, LOG_PATH.exists())

=== Calibration Analysis Saved ===
calibration table: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_final_uncertainty_calibration_outputs\selected_model_calibration_table.csv True
calibration summary: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_final_uncertainty_calibration_outputs\selected_model_calibration_summary.csv True
participant calibration: data\processed\pre_sleep_forecasting\design_c_stage1\experiments\stage1_final_uncertainty_calibration_outputs\selected_model_participant_calibration.csv True

=== Calibration Summary ===


,metric,value
0,samples,881.000000
1,participants,14.000000
2,bins,10.000000
3,brier_score,0.212638
4,expected_calibration_error,0.125631
5,maximum_calibration_error,0.834234
6,mean_predicted_probability,0.439648
7,observed_positive_rate,0.360953
8,selected_threshold,0.540000



=== Calibration Table ===


,probability_bin,samples,mean_predicted_probability,observed_positive_rate,predicted_positive_rate_at_threshold,abs_calibration_error
0,"(-0.001, 0.1]",0,NaN,NaN,NaN,NaN
1,"(0.1, 0.2]",2,0.165766,1.000000,0.000000,0.834234
2,"(0.2, 0.3]",62,0.274573,0.322581,0.000000,0.048007
3,"(0.3, 0.4]",344,0.350905,0.223837,0.000000,0.127068
4,"(0.4, 0.5]",191,0.445202,0.308901,0.000000,0.136302
5,"(0.5, 0.6]",180,0.545893,0.433333,0.577778,0.112560
6,"(0.6, 0.7]",95,0.638268,0.800000,1.000000,0.161732
7,"(0.7, 0.8]",5,0.739239,0.800000,1.000000,0.060761
8,"(0.8, 0.9]",2,0.818879,1.000000,1.000000,0.181121
9,"(0.9, 1.0]",0,NaN,NaN,NaN,NaN



=== Participant Calibration Summary ===


,participant_object_id,samples,observed_positive_rate,mean_predicted_probability,brier_score,abs_mean_probability_error
11,621e342e67b776a2404ce460,37,0.054054,0.450492,0.210467,0.396438
9,621e32e667b776a2406d2f1c,90,0.222222,0.526210,0.261360,0.303988
8,621e324e67b776a2400191cb,68,0.102941,0.376562,0.171738,0.273621
13,621e366567b776a24076a727,62,0.080645,0.314081,0.126435,0.233435
12,621e351a67b776a240f6204b,104,0.173077,0.391940,0.189677,0.218863
10,621e337667b776a240ce78ab,82,0.731707,0.530555,0.197167,0.201153
3,621e2ff067b776a2403eb737,60,0.716667,0.519048,0.207770,0.197619
2,621e2fb367b776a24015accd,59,0.254237,0.360955,0.213227,0.106717
4,621e301e67b776a240608a72,53,0.452830,0.364146,0.290998,0.088684
1,621e2f6167b776a240e082a9,45,0.288889,0.365306,0.219603,0.076418



log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True


In [4]:
# Cell 4. Update final report with bootstrap uncertainty and calibration results
# 원하는 결과:
# - 기존 pre-sleep final report에 participant bootstrap CI와 calibration 결과를 추가한다.
# - 모델의 신뢰 가능성과 확률 해석 한계를 명확히 쓴다.
# - updated report를 저장하고 log를 업데이트한다.

BASE_REPORT_PATH = PROJECT_ROOT / "reports" / "pre_sleep_forecasting_stage1_final_report.md"
UPDATED_REPORT_PATH = PROJECT_ROOT / "reports" / "pre_sleep_forecasting_stage1_final_report_updated.md"

bootstrap_summary_df = pd.read_csv(BOOTSTRAP_SUMMARY_PATH, encoding="utf-8-sig")
calibration_summary_df = pd.read_csv(CALIBRATION_SUMMARY_PATH, encoding="utf-8-sig")
calibration_table_df = pd.read_csv(CALIBRATION_TABLE_PATH, encoding="utf-8-sig")
participant_calibration_df = pd.read_csv(PARTICIPANT_CALIBRATION_PATH, encoding="utf-8-sig")

base_report = BASE_REPORT_PATH.read_text(encoding="utf-8")

def metric_summary(metric_name):
    row = bootstrap_summary_df[bootstrap_summary_df["metric"] == metric_name].iloc[0]
    return row

ba_ci = metric_summary("balanced_accuracy")
roc_ci = metric_summary("roc_auc")
ap_ci = metric_summary("average_precision")
f1_ci = metric_summary("f1")
precision_ci = metric_summary("precision")
recall_ci = metric_summary("recall")
brier_ci = metric_summary("brier_score")

calibration_values = dict(
    zip(calibration_summary_df["metric"], calibration_summary_df["value"])
)

worst_participant_calibration = (
    participant_calibration_df
    .sort_values("abs_mean_probability_error", ascending=False)
    .head(5)
)

uncertainty_and_calibration_section = f"""

## 11. Participant-Level Uncertainty

Participant-level bootstrap was performed on the held-out test participants.

- Bootstrap unit: participant
- Test participants: {int(calibration_values['participants'])}
- Bootstrap iterations: {BOOTSTRAP_ITERATIONS}
- Official threshold: {SELECTED_THRESHOLD:.2f}

Bootstrap confidence intervals:

| Metric | Point estimate | Bootstrap mean | 95% CI |
|---|---:|---:|---:|
| Balanced accuracy | {ba_ci['point_estimate']:.4f} | {ba_ci['bootstrap_mean']:.4f} | [{ba_ci['ci_lower_2_5']:.4f}, {ba_ci['ci_upper_97_5']:.4f}] |
| ROC AUC | {roc_ci['point_estimate']:.4f} | {roc_ci['bootstrap_mean']:.4f} | [{roc_ci['ci_lower_2_5']:.4f}, {roc_ci['ci_upper_97_5']:.4f}] |
| Average precision | {ap_ci['point_estimate']:.4f} | {ap_ci['bootstrap_mean']:.4f} | [{ap_ci['ci_lower_2_5']:.4f}, {ap_ci['ci_upper_97_5']:.4f}] |
| F1 | {f1_ci['point_estimate']:.4f} | {f1_ci['bootstrap_mean']:.4f} | [{f1_ci['ci_lower_2_5']:.4f}, {f1_ci['ci_upper_97_5']:.4f}] |
| Precision | {precision_ci['point_estimate']:.4f} | {precision_ci['bootstrap_mean']:.4f} | [{precision_ci['ci_lower_2_5']:.4f}, {precision_ci['ci_upper_97_5']:.4f}] |
| Recall | {recall_ci['point_estimate']:.4f} | {recall_ci['bootstrap_mean']:.4f} | [{recall_ci['ci_lower_2_5']:.4f}, {recall_ci['ci_upper_97_5']:.4f}] |
| Brier score | {brier_ci['point_estimate']:.4f} | {brier_ci['bootstrap_mean']:.4f} | [{brier_ci['ci_lower_2_5']:.4f}, {brier_ci['ci_upper_97_5']:.4f}] |

Interpretation:

- The balanced accuracy point estimate is {ba_ci['point_estimate']:.4f}.
- The participant-level 95% CI is [{ba_ci['ci_lower_2_5']:.4f}, {ba_ci['ci_upper_97_5']:.4f}].
- The CI is fairly wide because the held-out test set contains only {int(calibration_values['participants'])} participants.
- The model shows useful predictive signal, but uncertainty remains material.

## 12. Calibration Analysis

Calibration was evaluated on the held-out test split.

Calibration summary:

- Brier score: {calibration_values['brier_score']:.4f}
- Expected calibration error: {calibration_values['expected_calibration_error']:.4f}
- Maximum calibration error: {calibration_values['maximum_calibration_error']:.4f}
- Mean predicted probability: {calibration_values['mean_predicted_probability']:.4f}
- Observed positive rate: {calibration_values['observed_positive_rate']:.4f}

The model tends to overestimate the probability of good sleep overall. The mean predicted probability is {calibration_values['mean_predicted_probability']:.4f}, while the observed positive rate is {calibration_values['observed_positive_rate']:.4f}.

Participant-level calibration also varies substantially. The largest participant-level mean probability errors were:

| Participant | Samples | Observed positive rate | Mean predicted probability | Absolute error |
|---|---:|---:|---:|---:|
"""

for _, row in worst_participant_calibration.iterrows():
    uncertainty_and_calibration_section += (
        f"| `{row['participant_object_id']}` | "
        f"{int(row['samples'])} | "
        f"{row['observed_positive_rate']:.4f} | "
        f"{row['mean_predicted_probability']:.4f} | "
        f"{row['abs_mean_probability_error']:.4f} |\n"
    )

uncertainty_and_calibration_section += """

Calibration interpretation:

- The probability scores should not yet be interpreted as well-calibrated real-world probabilities.
- The current model is more appropriate for ranking and threshold-based classification than direct probability communication.
- Calibration correction may be explored later, but the validation participant count is small, so calibration models may overfit.

## 13. Updated Reliability Assessment

The current best model is useful as a research-grade strict pre-sleep forecasting candidate.

Strengths:

- Uses only data available before sleep onset.
- Uses a participant-level held-out test split.
- Has improved seed stability after hyperparameter tuning.
- Has participant-level uncertainty estimates.
- Outperforms previous pre-sleep and rolling/history alternatives by balanced accuracy.

Limitations:

- Bootstrap confidence intervals remain wide.
- Test set has only 14 participants.
- Recall is moderate under the official threshold.
- Probability calibration is imperfect.
- External validation is still absent.

Recommended use:

- Suitable for exploratory personal sleep-quality feedback research.
- Suitable for comparing relative risk/likelihood across nights.
- Not yet suitable for high-stakes clinical or medical decision-making.
- Probability values should be communicated cautiously unless calibration is improved.
"""

updated_report = base_report + uncertainty_and_calibration_section

UPDATED_REPORT_PATH.write_text(updated_report, encoding="utf-8")

log_entry = f"""

## 2026-06-29 - Updated pre-sleep forecasting final report with uncertainty/calibration

Updated final pre-sleep forecasting report with participant bootstrap CI and calibration results.

- Base report: `{BASE_REPORT_PATH.relative_to(PROJECT_ROOT)}`
- Updated report: `{UPDATED_REPORT_PATH.relative_to(PROJECT_ROOT)}`
- Balanced accuracy bootstrap 95% CI: [{ba_ci['ci_lower_2_5']:.4f}, {ba_ci['ci_upper_97_5']:.4f}]
- Brier score: {calibration_values['brier_score']:.4f}
- Expected calibration error: {calibration_values['expected_calibration_error']:.4f}
- Main reliability note: predictive signal is useful but uncertainty and calibration limitations remain.
"""

with LOG_PATH.open("a", encoding="utf-8") as f:
    f.write(log_entry)

print("updated report written:", UPDATED_REPORT_PATH.relative_to(PROJECT_ROOT), UPDATED_REPORT_PATH.exists())
print("updated report characters:", len(updated_report))
print("log updated:", LOG_PATH, LOG_PATH.exists())

updated report written: reports\pre_sleep_forecasting_stage1_final_report_updated.md True
updated report characters: 9654
log updated: C:\workSpace\DeepLearnin_sleep\log\2026-06-29.md True
